# XGBoost and LightGBM Ensemble Regression
This notebook processes tabular data to predict the number of calories burned during exercise sessions.
The task is a supervised regression problem evaluated using the Root Mean Squared Logarithmic Error (RMSLE).

We will test two models and blend their predictions:
1. XGBoost Regressor
2. LightGBM Regressor

In [ ]:
!pip install -r requirements.txt

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import lightgbm as lgb

## 1. Load the Data
We load the `train.csv` and `test.csv` datasets. The training set contains entries for `id`, `Sex`, `Age`, `Height`, `Weight`, `Duration`, `Heart_Rate`, `Body_Temp`, and `Calories`.

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
train.head()

Train shape : (750000, 9)
Test shape  : (250000, 8)


,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


## 2. Preprocessing and Feature Engineering
We encode the categorical `Sex` column and engineer interaction and polynomial features that capture non-linear relationships between exercise intensity, duration, and calorie output.
Because the evaluation metric is RMSLE, we apply `log1p` to the target before training. This is mathematically equivalent to directly minimizing RMSLE.

In [3]:
def make_features(df):
    df = df.copy()

    # Encode sex
    df["Sex"] = (df["Sex"] == "male").astype(int)

    # Body metrics
    df["BMI"]      = df["Weight"] / (df["Height"] / 100) ** 2
    df["Height_m"] = df["Height"] / 100

    # Interaction features
    df["Duration_x_HR"]      = df["Duration"] * df["Heart_Rate"]
    df["Duration_x_BT"]      = df["Duration"] * df["Body_Temp"]
    df["HR_x_BT"]            = df["Heart_Rate"] * df["Body_Temp"]
    df["Duration_x_HR_x_BT"] = df["Duration"] * df["Heart_Rate"] * df["Body_Temp"]

    # Polynomial features
    df["Duration2"]  = df["Duration"] ** 2
    df["HeartRate2"] = df["Heart_Rate"] ** 2
    df["BodyTemp2"]  = df["Body_Temp"] ** 2

    # Ratio features
    df["Duration_per_Weight"] = df["Duration"] / df["Weight"]
    df["HR_per_Age"]          = df["Heart_Rate"] / df["Age"]

    # Age group buckets
    df["Age_group"] = pd.cut(
        df["Age"], bins=[0, 30, 45, 60, 100], labels=[0, 1, 2, 3]
    ).astype(int)

    return df.drop(columns=["id"])


FEATURE_COLS = [c for c in make_features(train).columns if c != "Calories"]

X_train = make_features(train)[FEATURE_COLS]
X_test  = make_features(test)[FEATURE_COLS]

y_train_raw = train["Calories"]
y_train_log = np.log1p(train["Calories"])

print(f"Feature count : {len(FEATURE_COLS)}")
print(f"Features      : {FEATURE_COLS}")

Feature count : 19
Features      : ['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'BMI', 'Height_m', 'Duration_x_HR', 'Duration_x_BT', 'HR_x_BT', 'Duration_x_HR_x_BT', 'Duration2', 'HeartRate2', 'BodyTemp2', 'Duration_per_Weight', 'HR_per_Age', 'Age_group']


## 3. Evaluation Metric and Cross-Validation Setup
We define the RMSLE function used to score predictions throughout the notebook.
A 5-fold cross-validation strategy is used so that every training row appears in a validation fold exactly once, giving us a reliable out-of-fold (OOF) estimate of leaderboard performance.

In [4]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))


N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_xgb  = np.zeros(len(X_train))
oof_lgbm = np.zeros(len(X_train))
test_xgb  = np.zeros(len(X_test))
test_lgbm = np.zeros(len(X_test))

## 4. Train XGBoost
The model is trained on the log-transformed target with early stopping on each fold. Out-of-fold predictions are stored in raw scale (after `expm1`) for final RMSLE scoring.

In [ ]:
xgb_params = dict(
    n_estimators          = 3000,
    learning_rate         = 0.01,
    max_depth             = 8,
    min_child_weight      = 3,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    colsample_bylevel     = 0.8,
    gamma                 = 0.01,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    n_jobs                = -1,
    verbosity             = 0,
    random_state          = 42,
    early_stopping_rounds = 150,
    eval_metric           = "rmse",
)

xgb_fold_rmsle = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val   = X_train.iloc[tr_idx],    X_train.iloc[val_idx]
    y_tr, y_val   = y_train_log.iloc[tr_idx], y_train_log.iloc[val_idx]
    y_val_raw     = y_train_raw.iloc[val_idx]

    model = XGBRegressor(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    val_preds_raw  = np.expm1(model.predict(X_val))
    test_preds_raw = np.expm1(model.predict(X_test))

    oof_xgb[val_idx]  = val_preds_raw
    test_xgb         += test_preds_raw / N_FOLDS

    score = rmsle(y_val_raw, val_preds_raw)
    xgb_fold_rmsle.append(score)
    print(f"  Fold {fold+1}/{N_FOLDS}  |  RMSLE: {score:.5f}  |  best_iter: {model.best_iteration}")

xgb_oof_rmsle = rmsle(y_train_raw, oof_xgb)
print(f"\nXGBoost OOF RMSLE: {xgb_oof_rmsle:.5f}")

  Fold 1/5  |  RMSLE: 0.05951  |  best_iter: 1661
  Fold 2/5  |  RMSLE: 0.06041  |  best_iter: 1403
  Fold 3/5  |  RMSLE: 0.05945  |  best_iter: 1644
  Fold 4/5  |  RMSLE: 0.05988  |  best_iter: 1700


## 5. Train LightGBM
The same log1p target strategy is applied. LightGBM uses leaf-wise tree growth which often converges to a lower error than XGBoost on large datasets.

In [ ]:
lgbm_params = dict(
    n_estimators      = 3000,
    learning_rate     = 0.01,
    num_leaves        = 256,
    max_depth         = -1,
    min_child_samples = 20,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    n_jobs            = -1,
    verbosity         = -1,
    random_state      = 42,
)

lgbm_fold_rmsle = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val   = X_train.iloc[tr_idx],    X_train.iloc[val_idx]
    y_tr, y_val   = y_train_log.iloc[tr_idx], y_train_log.iloc[val_idx]
    y_val_raw     = y_train_raw.iloc[val_idx]

    model = lgb.LGBMRegressor(**lgbm_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    val_preds_raw  = np.expm1(model.predict(X_val))
    test_preds_raw = np.expm1(model.predict(X_test))

    oof_lgbm[val_idx]  = val_preds_raw
    test_lgbm         += test_preds_raw / N_FOLDS

    score = rmsle(y_val_raw, val_preds_raw)
    lgbm_fold_rmsle.append(score)
    print(f"  Fold {fold+1}/{N_FOLDS}  |  RMSLE: {score:.5f}  |  best_iter: {model.best_iteration_}")

lgbm_oof_rmsle = rmsle(y_train_raw, oof_lgbm)
print(f"\nLightGBM OOF RMSLE: {lgbm_oof_rmsle:.5f}")

## 6. Ensemble
We blend the two models' out-of-fold predictions to find the weight that minimizes OOF RMSLE, then apply that same weight to the test predictions.

In [ ]:
best_w, best_rmsle = 0.5, 999.0

for w in np.arange(0.0, 1.01, 0.05):
    blended = w * oof_xgb + (1 - w) * oof_lgbm
    score   = rmsle(y_train_raw, blended)
    if score < best_rmsle:
        best_rmsle, best_w = score, w

final_test_preds = np.maximum(best_w * test_xgb + (1 - best_w) * test_lgbm, 0)

print("-" * 40)
print(f"XGBoost  OOF RMSLE : {xgb_oof_rmsle:.5f}")
print(f"LightGBM OOF RMSLE : {lgbm_oof_rmsle:.5f}")
print(f"Ensemble OOF RMSLE : {best_rmsle:.5f}  (XGB={best_w:.0%} / LGBM={(1-best_w):.0%})")
print("-" * 40)

## 7. Generate Submission
We format the final predictions and save them to `submission.csv` for upload to Kaggle.

In [ ]:
submission = pd.DataFrame({
    "id":       test["id"],
    "Calories": final_test_preds,
})

submission.to_csv("submission.csv", index=False)
print(f"Successfully created submission.csv from test.csv!")
print(submission.head())